# 01 - Exploration et preparation des donnees


Membres : Nom Prenom - CIP

Objectif : decrire le jeu de donnees, verifier s'il necessite un nettoyage,
et constituer l'ensemble de depart de 750 variables.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")

import numpy as np
import pandas as pd

## Description du jeu de donnees

In [ ]:
X = pd.read_csv("../data/data.csv", index_col=0)
y = pd.read_csv("../data/labels.csv", index_col=0)["Class"]

print(X.shape)
y.value_counts()

## Nettoyage des donnees

Le Theme 1 (diapositives 23 a 31) traite des donnees manquantes, des doublons
et des valeurs aberrantes. On verifie d'abord si ce nettoyage est necessaire ici.

In [ ]:
print("valeurs manquantes :", X.isna().sum().sum())
print("patients en double :", X.index.duplicated().sum())
print("patients alignes avec les etiquettes :", (X.index == y.index).all())
print("intervalle des valeurs : %.2f a %.2f" % (X.values.min(), X.values.max()))
print("proportion de zeros : %.1f%%" % (100 * (X.values == 0).mean()))

Aucune valeur manquante, aucun doublon, et les deux fichiers sont alignes :
les techniques d'imputation du Theme 1 ne s'appliquent pas a ce jeu de donnees.

Les valeurs sont des expressions genetiques deja transformees en echelle
logarithmique, toutes positives. Les zeros correspondent a des genes non
exprimes : ce sont de vraies mesures, pas des donnees manquantes.

Les genes de variance nulle, en revanche, doivent etre retires : ils ne
contribuent a aucune distance et rendent la matrice de covariance singuliere,
ce qui empeche le calcul de la distance de Mahalanobis.

In [ ]:
nulles = (X.var() == 0).sum()
print("genes de variance nulle :", nulles)
print("genes restants :", X.shape[1] - nulles)

## Echelle et correlation

Ces deux proprietes justifient des choix faits plus loin : la standardisation
avant l'ACP, et l'interet de la distance de Mahalanobis.

In [ ]:
Xr = X.loc[:, X.var() > 0]

s = Xr.std()
print("ecarts-type : min %.2f  max %.2f  (rapport x%.0f)" % (s.min(), s.max(), s.max() / s.min()))

# Correlation sur un echantillon de genes : la matrice complete serait trop grande.
ech = Xr.sample(500, axis=1, random_state=0)
C = np.corrcoef(ech, rowvar=False)
hors_diag = C[~np.eye(500, dtype=bool)]
print("correlation moyenne (valeur absolue) : %.3f" % np.abs(hors_diag).mean())
print("paires de genes avec |r| > 0.5 : %d" % ((np.abs(hors_diag) > 0.5).sum() // 2))

Les ecarts-type varient d'un facteur important : l'ACP sera donc appliquee sur
des variables standardisees (diapositives 41 et 42).

De nombreuses paires de genes sont fortement correlees, ce qui justifie l'emploi
de la distance de Mahalanobis, qui tient compte de cette structure de covariance.

## Ensemble de depart

L'enonce (section 2.2) demande de reduire la dimensionnalite des le depart, par
une selection qui peut etre aleatoire. On tire 750 genes au hasard parmi les
20264 restants (seed 42, voir `src/config.py`).

Ce choix est motive par le cout de calcul : l'ACP et surtout la recherche de
paires de variables de la Methode 2 sont quadratiques en nombre de variables.

Il faut aussi verifier que la matrice de covariance reste inversible, sans quoi
la distance de Mahalanobis ne peut pas etre calculee.

In [ ]:
from src.loading import preparer

preparer()

In [ ]:
from src.loading import charger

X_sub, y_arr = charger()
print(X_sub.shape)

In [ ]:
from src.config import CLASSES

# Sigma poolee : une seule matrice estimee sur les 801 patients.
S = np.cov(X_sub, rowvar=False)
print("Sigma poolee : rang %d sur %d" % (np.linalg.matrix_rank(S), S.shape[0]))

# A titre de comparaison, une Sigma estimee classe par classe.
for c in CLASSES:
    Xc = X_sub[y_arr == c]
    Sc = np.cov(Xc, rowvar=False)
    print("  %s (%3d patients) : rang %3d sur %d" % (
        c, len(Xc), np.linalg.matrix_rank(Sc), Sc.shape[0]))

La matrice de covariance poolee, estimee sur les 801 patients, est de rang plein :
elle est inversible et la distance de Mahalanobis peut etre calculee sur les
750 variables.

Une matrice estimee classe par classe serait en revanche singuliere, car chaque
classe compte moins de 750 patients (la plus petite, COAD, n'en compte que 78).
C'est la raison pour laquelle nous utilisons une covariance poolee plutot qu'une
covariance par classe : elle permet de conserver l'ensemble des 750 variables
pour les deux metriques, et la comparaison Euclidienne / Mahalanobis porte donc
exactement sur les memes donnees.

### Commentaires

Les donnees ne necessitent aucun nettoyage : pas de valeur manquante, pas de
doublon, et les deux fichiers sont alignes. Les 14 % de zeros sont des genes non
exprimes, pas des donnees manquantes.

Les 267 genes de variance nulle sont retires : ils ne contribuent a aucune
distance et rendent la matrice de covariance singuliere.

Le nombre de variables est limite a 750 car la covariance est estimee sur 801
patients : au-dela de 800 variables elle devient singuliere et Mahalanobis ne
peut plus etre calculee.

Le calcul de rang confirme ce choix : la covariance poolee est de rang plein
(750/750), alors que les covariances par classe sont toutes singulieres
(rang = nombre de patients - 1).